
# NLP Assignment
Part 1 demonstrates:
1. Downloading a small English corpus (NLTK Movie Reviews)
2. Preprocessing: tokenization, lowercasing, HTML removal,
   punctuation removal, and lemmatization
3. Showing before/after samples
4. Training N-gram language models (N=2, 3, and 4)
5. Generating text from the model
6. Computing Perplexity for each model

In [33]:
## Import Statement
import nltk
import random
import re
import spacy   #  spaCy for lemmatization
import string
import numpy  as np
import pandas as pd

# For movie review data
from nltk.corpus import movie_reviews

# Preprocessing utilities
from bs4         import BeautifulSoup # To clean html
from nltk        import word_tokenize

# LM utilities
from collections import defaultdict, Counter # See comments below on usefulness of these utils
from math        import log, exp             # to compute perplexity
from nltk.util   import ngrams               # to build the N-Gram models

# Download required NLTK resources
nltk.download('movie_reviews')
nltk.download('punkt')
nltk.download('punkt_tab')

# Load spaCy English model
# If you get an error trying to run the next line of code
# then do this first (or do it in a terminal window):
# !python3 -m spacy download en_core_web_sm
nlp = spacy.load("en_core_web_sm")

'Ready'

[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


'Ready'

## **Observation**

**What this cell does**  
This cell imports the libraries used throughout the notebook, downloads required NLTK resources (movie_reviews, punkt, punkt_tab), and loads the spaCy English model en_core_web_sm.

**Output interpretation**

The NLTK downloader messages confirm the corpora and tokenizers were installed and unzipped successfully. The final printed value 'Ready' indicates the cell completed and the spaCy model loaded without error. If you see an error when loading spaCy, run python -m spacy download en_core_web_sm in the Colab cell and re-run this import cell.

In [34]:
# Load raw sentences from the corpus
fileids = movie_reviews.fileids()
sentences_raw = []

for fid in fileids:
    text = movie_reviews.raw(fid)
    # Split into sentences using NLTK's sentence tokenizer
    sents = nltk.sent_tokenize(text)
    sentences_raw.extend(sents)

print(f"Total sentences loaded: {len(sentences_raw)}")

Total sentences loaded: 71532


**Observation:** The movie reviews corpus was split into 71,532 raw sentences using NLTK's sentence tokenizer; this yields a large sentence-level dataset for training N‑gram models, though many sentences may be short or noisy and will be cleaned during preprocessing.

In [35]:
# Clean the data (preprocessing): remove HTML, lowercase, remove punctuation,
# tokenize, and lemmatize using spaCy. Produces `processed_sentences`.

import re
from bs4 import BeautifulSoup

# Regex to keep letters, numbers and spaces (remove punctuation)
_non_alnum_re = re.compile(r"[^a-z0-9\s']+")

def clean_text_for_lemmatization(text: str) -> str:
    """Remove HTML, lowercase, normalize whitespace, remove punctuation."""
    text = BeautifulSoup(text, "html.parser").get_text()        # strip HTML
    text = text.lower()                                        # lowercase
    text = text.replace("\n", " ").replace("\r", " ")          # normalize newlines
    text = _non_alnum_re.sub(" ", text)                        # remove punctuation (keep apostrophes optionally)
    text = re.sub(r"\s+", " ", text).strip()                   # collapse whitespace
    return text

processed_sentences = []
batch_size = 2000   # adjust for memory/speed tradeoff

# Use spaCy pipe for faster batch lemmatization
texts_to_process = []
orig_index_map = []   # keep mapping to original sentence index if needed

for i, s in enumerate(sentences_raw):
    s_clean = clean_text_for_lemmatization(s)
    if s_clean:                       # skip empty results
        texts_to_process.append(s_clean)
        orig_index_map.append(i)

# Process in batches to avoid memory spikes
for start in range(0, len(texts_to_process), batch_size):
    batch = texts_to_process[start:start+batch_size]
    for doc in nlp.pipe(batch, disable=["ner", "parser"]):
        # Keep lemma for each token; filter out spaces and punctuation tokens
        lemmas = [tok.lemma_ for tok in doc if not tok.is_space and not tok.is_punct]
        # Optionally filter tokens that are just apostrophes or empty
        lemmas = [t for t in lemmas if t and t != "'"]
        if lemmas:
            processed_sentences.append(" ".join(lemmas))

# Final sanity: remove any duplicates of empty strings and ensure alignment
processed_sentences = [s for s in processed_sentences if s.strip()]

print(f"Raw sentences: {len(sentences_raw)}")
print(f"Processed sentences: {len(processed_sentences)}")

# `processed_sentences` is now ready for downstream steps (train/test split, n-grams, etc.)


Raw sentences: 71532
Processed sentences: 67757


**Observation:** After cleaning and lemmatization the dataset reduced from 71,532 raw sentences to 67,757 processed sentences, indicating that ~3,775 sentences were dropped (empty or fully filtered) during normalization. The preprocessing (HTML removal, lowercasing, punctuation stripping, and spaCy lemmatization using batched nlp.pipe) produced cleaner, dictionary‑style sentence tokens suitable for N‑gram counting and reduced surface form variation, while batching avoided memory spikes.

In [36]:
# Display 5 sentences before and after preprocessing.

from pprint import pprint

n_show = 5

# Prefer aligned display using orig_index_map if available
try:
    # Show the first n_show processed sentences and their original raw sentences
    print("Showing first", n_show, "processed sentences with their original raw sentences:\n")
    for i in range(min(n_show, len(processed_sentences))):
        orig_idx = orig_index_map[i] if 'orig_index_map' in globals() else None
        raw = sentences_raw[orig_idx] if orig_idx is not None else sentences_raw[i]
        print(f"--- Example {i+1} ---")
        print("Raw:      ", raw)
        print("Processed:", processed_sentences[i])
        print()
except Exception as e:
    # Fallback: show first n_show of each list independently
    print("Could not align with orig_index_map (falling back to independent samples). Error:", e)
    print("\nFirst", n_show, "raw sentences:\n")
    for s in sentences_raw[:n_show]:
        print("-", s)
    print("\nFirst", n_show, "processed sentences:\n")
    for s in processed_sentences[:n_show]:
        print("-", s)


Showing first 5 processed sentences with their original raw sentences:

--- Example 1 ---
Raw:       plot : two teen couples go to a church party , drink and then drive .
Processed: plot two teen couple go to a church party drink and then drive

--- Example 2 ---
Raw:       they get into an accident .
Processed: they get into an accident

--- Example 3 ---
Raw:       one of the guys dies , but his girlfriend continues to see him in her life , and has nightmares .
Processed: one of the guy die but his girlfriend continue to see he in her life and have nightmare

--- Example 4 ---
Raw:       what's the deal ?
Processed: what be the deal

--- Example 5 ---
Raw:       watch the movie and " sorta " find out .
Processed: watch the movie and sorta find out



**Observation:** The side‑by‑side examples show the cleaning pipeline working as intended: HTML/punctuation removed, text lowercased, and spaCy lemmatization applied *(e.g., dies → die, what's → what be)*. The orig_index_map preserves alignment between raw and processed sentences so you can trace changes. Note that lemmatization and aggressive normalization reduce surface variation and prepare data for N‑gram counting, but they also alter some grammatical forms and pronouns, so expect small semantic shifts in a few examples.

In [37]:
# Split processed sentences into train/test (90% train, 10% test)

from sklearn.model_selection import train_test_split

# Ensure reproducibility
RANDOM_STATE = 42

# Remove any empty strings just in case
processed_sentences = [s for s in processed_sentences if s and s.strip()]

# 90% train, 10% test
train_sentences, test_sentences = train_test_split(
    processed_sentences,
    test_size=0.10,
    random_state=RANDOM_STATE,
    shuffle=True
)

print(f"Total processed sentences: {len(processed_sentences)}")
print(f"Training sentences: {len(train_sentences)}")
print(f"Test sentences: {len(test_sentences)}\n")

# Show 5 examples from each split
print("=== 5 training examples ===")
for i, s in enumerate(train_sentences[:5], 1):
    print(f"{i}. {s}")

print("\n=== 5 test examples ===")
for i, s in enumerate(test_sentences[:5], 1):
    print(f"{i}. {s}")


Total processed sentences: 67757
Training sentences: 60981
Test sentences: 6776

=== 5 training examples ===
1. there be of course non stop action in this movie everything from one on one kung fu to all out gun battle to a sppedboat with eight engine
2. carter kerr smith the hotshot jock hold he in contempt because he believe that it be only he himself who can decide his fate
3. no they want to see he do something that take a little more in the act department namely play baseball
4. simply irresistible on the other hand be about as challenging as an easy bake oven
5. a different mystery I guess which may turn into a murder mystery if that video ever come on again

=== 5 test examples ===
1. watch it with some friend for a good laugh and celebrate
2. in fact virus be yet another in the long line of action horror paranoia thriller from the alien vein a group of people be drop into a mysterious situation only to find a mortally threatening entity be out to get they a textbook example hail

**Observation**
The processed corpus was split into 90% training (60,981 sentences) and 10% test (6,776 sentences) using a fixed random seed for reproducibility; this preserves a large training set for N‑gram estimation while keeping a held‑out set for evaluation (generation and perplexity).

In [38]:
# Function(s) to generate sentences from an N-gram model with Laplace (add-one) smoothing.

import random
import math
from typing import Tuple, List, Set, Dict
from collections import Counter, defaultdict

def _laplace_prob(context: Tuple[str, ...],
                  word: str,
                  ngram_counts: Dict[Tuple[str, ...], Counter],
                  context_counts: Counter,
                  V: int) -> float:
    """Return P(word | context) with add-one (Laplace) smoothing."""
    return (ngram_counts[context].get(word, 0) + 1) / (context_counts.get(context, 0) + V)

def generate_sentence(N, ngram_counts, context_counts, vocab, max_len=30, rng=None):
    """
    Generate a single sentence using observed-continuation sampling with simple fallback.
    - Samples from observed continuations for the current context (with add-one smoothing).
    - If the context has no observed continuations, falls back to uniform sampling over vocab.
    """
    if rng is None:
        rng = random

    V = len(vocab)
    context = tuple(["<s>"] * (N - 1))
    generated = []

    for _ in range(max_len):
        cont_counts = ngram_counts.get(context, Counter())
        observed = list(cont_counts.keys())

        if observed:
            # Compute add-one smoothed probabilities only over observed continuations
            # p(w|context) = (count(context->w) + 1) / (context_count + V)
            # We compute numerator for observed words and normalize.
            numerators = [(cont_counts[w] + 1) for w in observed]
            denom = context_counts.get(context, 0) + V
            probs = [num / denom for num in numerators]
            # Normalize to sum to 1 (numerical safety)
            total = sum(probs)
            probs = [p / total for p in probs]
            next_word = rng.choices(observed, weights=probs, k=1)[0]
        else:
            # Unseen context: simple fallback/backoff
            # Option A (simple): uniform over vocab
            next_word = rng.choice(list(vocab))

        if next_word == "</s>":
            break
        generated.append(next_word)

        if N > 1:
            context = tuple(list(context[1:]) + [next_word])

    return " ".join(generated)


**Observation**
The functions implement sentence generation from an N‑gram model using Laplace smoothing and optional seeding for reproducibility; they sample next tokens from the full vocabulary conditioned on the current context and stop at </s> or max_len. Use these functions after building ngram_counts, context_counts, and vocab from your training data.

In [39]:
# Perplexity evaluation utilities for N-gram models with Laplace smoothing

import math
from nltk.util import ngrams
from collections import Counter
from typing import Tuple, Dict, Set, List

def laplace_prob(context: Tuple[str, ...],
                 word: str,
                 ngram_counts: Dict[Tuple[str, ...], Counter],
                 context_counts: Counter,
                 V: int) -> float:
    """Return P(word | context) with add-one (Laplace) smoothing."""
    return (ngram_counts.get(context, Counter()).get(word, 0) + 1) / (context_counts.get(context, 0) + V)

def sentence_logprob(sentence: str,
                     N: int,
                     ngram_counts: Dict[Tuple[str, ...], Counter],
                     context_counts: Counter,
                     V: int,
                     vocab: Set[str] = None) -> Tuple[float, int]:
    """
    Compute log-probability of a single sentence under an N-gram model.
    Unseen tokens are mapped to <UNK> if vocab is provided.
    """
    # Map unseen tokens to <UNK> if vocab provided
    tokens_raw = sentence.split()
    if vocab is not None:
        tokens_mapped = [t if t in vocab else "<UNK>" for t in tokens_raw]
    else:
        tokens_mapped = tokens_raw

    tokens = ["<s>"] * (N - 1) + tokens_mapped + ["</s>"]
    logp = 0.0
    T = 0
    for ng in ngrams(tokens, N):
        context = tuple(ng[:-1])
        word = ng[-1]
        p = laplace_prob(context, word, ngram_counts, context_counts, V)
        logp += math.log(p)
        T += 1
    return logp, T


def perplexity(test_sentences: List[str],
               N: int,
               ngram_counts: Dict[Tuple[str, ...], Counter],
               context_counts: Counter,
               vocab: Set[str]) -> float:
    V = len(vocab)
    total_logp = 0.0
    total_T = 0
    for s in test_sentences:
        lp, T = sentence_logprob(s, N, ngram_counts, context_counts, V, vocab=vocab)
        total_logp += lp
        total_T += T
    if total_T == 0:
        return float('inf')
    return math.exp(- total_logp / total_T)


**Observation**
The functions implement sentence generation from an N‑gram model using Laplace smoothing and optional seeding for reproducibility; they sample next tokens from the full vocabulary conditioned on the current context and stop at </s> or max_len. Use these functions after building ngram_counts, context_counts, and vocab from your training data.

### Answers to Part 1 conceptual questions

**A. Context growth and data needs**  
As \(N\) increases the number of possible contexts grows roughly as \(|V|^{N-1}\). In our run the number of distinct contexts rose from ~30k (bigrams) to ~373k (trigrams) to ~800k (4‑grams), so many contexts have very few or zero observations. Larger N therefore requires much more data to estimate conditional probabilities reliably because each distinct context needs sufficient counts.

**B. Data sparsity effects**  
Data sparsity means most high‑order contexts are rare or unseen, so conditional probabilities estimated from counts are noisy or zero. In our results this produced much higher perplexity for trigram/4‑gram models: unseen or near‑zero counts force smoothing to dominate, which inflates perplexity.

**C. Why smoothing matters more for larger N**  
Smoothing redistributes probability mass to unseen events. For large N the fraction of unseen events is much higher, so naive add‑one smoothing becomes both necessary and insufficient: it assigns too much mass to the huge tail of unseen tokens and yields poor estimates. Better options are backoff/Kneser‑Ney or pruning the vocabulary/context counts.


In [40]:
from collections import defaultdict, Counter
from nltk.util import ngrams
import random
import math

def build_ngram_counts(sentences, N):
    ngram_counts = defaultdict(Counter)
    context_counts = Counter()
    vocab = set()
    for sent in sentences:
        tokens = ["<s>"] * (N - 1) + sent.split() + ["</s>"]
        vocab.update(tokens)
        for ng in ngrams(tokens, N):
            context = tuple(ng[:-1])
            word = ng[-1]
            ngram_counts[context][word] += 1
            context_counts[context] += 1
    return ngram_counts, context_counts, vocab

def _laplace_prob(context, word, ngram_counts, context_counts, V):
    return (ngram_counts.get(context, Counter()).get(word, 0) + 1) / (context_counts.get(context, 0) + V)

def generate_sentences(N, ngram_counts, context_counts, vocab, num_sentences=5, max_len=30, seed=None):
    rng = random.Random(seed) if seed is not None else None
    return [generate_sentence(N, ngram_counts, context_counts, vocab, max_len, rng) for _ in range(num_sentences)]

models = {}
for N in (2, 3, 4):
    ngram_counts, context_counts, vocab = build_ngram_counts(train_sentences, N)
    # Ensure an explicit unknown token exists in the vocabulary
    vocab.add("<UNK>")
    # (Optional) ensure contexts include <UNK> counts of zero by leaving ngram_counts unchanged
    models[N] = {"ngram_counts": ngram_counts, "context_counts": context_counts, "vocab": vocab}
    print(f"Built {N}-gram model: contexts={len(context_counts):,}, vocab={len(vocab):,}")

SEED = 42
NUM_GEN = 5

for N in (2, 3, 4):
    print("\n" + "="*60)
    print(f"{N}-GRAM MODEL (N={N})")
    ngram_counts = models[N]["ngram_counts"]
    context_counts = models[N]["context_counts"]
    vocab = models[N]["vocab"]

    print(f"\nGenerated {NUM_GEN} sentences (seed={SEED}):")
    gens = generate_sentences(N, ngram_counts, context_counts, vocab, num_sentences=NUM_GEN, max_len=30, seed=SEED)
    for i, s in enumerate(gens, 1):
        print(f"{i}. {s}")

    pp = perplexity(test_sentences, N, ngram_counts, context_counts, vocab)
    print(f"\nPerplexity (N={N}): {pp:.4f}")

print("\nDone.")


Built 2-gram model: contexts=30,214, vocab=30,216
Built 3-gram model: contexts=373,451, vocab=30,216
Built 4-gram model: contexts=800,284, vocab=30,216

2-GRAM MODEL (N=2)

Generated 5 sentences (seed=42):
1. that it be also he try vainly to go to the parting of the horse shoot an unlikely romance
2. barrymore whose collection of isaac asimov it be achieve with utter incompetence that gere be what surprise in scope out through the new account for a good as 1984 la
3. now the point along and point of patti 's clothing or environment race african go off the film star jet li and psychological drama occur in flight and still alive
4. all
5. 1998 tap on the rebs in the film be many small gap in a vhs

Perplexity (N=2): 2174.9581

3-GRAM MODEL (N=3)

Generated 5 sentences (seed=42):
1. that be never give enough to resort to hand
2. the happy bastard 's quick movie review wild wild west collapse under it be clueless as to why and of the strained awkward conversation between the bruce willis a

**Observation**
Add this observation as a short markdown cell below the code cell:  
The three N‑gram models were built from the training split; vocabulary size is large (~30k tokens) and context counts grow quickly with higher N (2‑gram ≈ 30k contexts, 3‑gram ≈ 373k, 4‑gram ≈ 800k). Generated samples show many rare or noisy tokens and limited fluency because generation samples from the full vocabulary with Laplace smoothing; perplexities are very high (bigram lower than trigram/4‑gram here) indicating data sparsity and the blunt effect of add‑one smoothing over a large vocabulary. Recommendations: reduce vocabulary (min frequency cutoff), use better smoothing/backoff (e.g., Kneser‑Ney or Katz backoff), or restrict sampling to observed continuations of a context (with a small probability mass for unknowns) to produce more coherent generated sentences and lower perplexity.

In [41]:
# ---------------------------------------------------------
# NLP Assignment 1 Part 2  75 Points
# ---------------------------------------------------------

import nltk
from nltk.corpus import movie_reviews
import random
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Ensure corpus is downloaded
nltk.download('movie_reviews')

'Movie Reviews Ready'

[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!


'Movie Reviews Ready'

**Observation**
The required libraries and evaluation tools are imported and the movie_reviews corpus is downloaded; the printed confirmation indicates the corpus is available for subsequent data loading and processing.

In [42]:
# Collect documents and labels
documents = []
labels = []

for category in movie_reviews.categories():
    for fid in movie_reviews.fileids(category):
        text = movie_reviews.raw(fid)
        documents.append(text)
        labels.append(category)

# Convert to DataFrame for convenience
df = pd.DataFrame({"text": documents, "label": labels})

print(df.head())
print(df['label'].value_counts())


                                                text label
0  plot : two teen couples go to a church party ,...   neg
1  the happy bastard's quick movie review \ndamn ...   neg
2  it is movies like these that make a jaded movi...   neg
3   " quest for camelot " is warner bros . ' firs...   neg
4  synopsis : a mentally unstable man undergoing ...   neg
label
neg    1000
pos    1000
Name: count, dtype: int64


**Observation:** The corpus was loaded into a DataFrame with 2,000 documents evenly split between classes (1000 neg, 1000 pos). This provides a balanced dataset for supervised experiments; the raw review text is preserved in the text column and the sentiment label in label, making it ready for vectorization and model training.

In [43]:
# Create train/test splits for classification (80% train, 20% test)

RANDOM_STATE = 42
TEST_SIZE = 0.20

# Ensure no missing values
df = df.dropna(subset=["text", "label"]).reset_index(drop=True)

# Stratify to preserve class balance
X_train, X_test, y_train, y_test = train_test_split(
    df["text"],
    df["label"],
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df["label"]
)

print(f"Total documents: {len(df)}")
print(f"Training documents: {len(X_train)}")
print(f"Test documents: {len(X_test)}\n")

print("Training label distribution:")
print(y_train.value_counts(), "\n")

print("Test label distribution:")
print(y_test.value_counts())


Total documents: 2000
Training documents: 1600
Test documents: 400

Training label distribution:
label
pos    800
neg    800
Name: count, dtype: int64 

Test label distribution:
label
neg    200
pos    200
Name: count, dtype: int64


**Observation:** The dataset was split into 1,600 training and 400 test documents with stratified sampling preserving class balance (800/800 train, 200/200 test); this ensures reproducible, balanced evaluation for downstream vectorization and classification.

In [44]:
# TF-IDF vectorizer: convert text to weighted word features
# Assumes X_train, X_test (pandas Series of raw text) are defined from previous cell.

from sklearn.feature_extraction.text import TfidfVectorizer

# Configuration
MAX_FEATURES = 20000   # adjust if notebook specifies a different value
NGRAM_RANGE = (1, 2)   # unigrams; change to (1,2) to include bigrams
STOP_WORDS = "english"
RANDOM_STATE = 42

# Initialize vectorizer (lowercase=True by default)
tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words=STOP_WORDS,
    max_features=MAX_FEATURES,
    ngram_range=NGRAM_RANGE,
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

# Fit on training data and transform both train and test
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

# Quick diagnostics
print("TF-IDF fitted.")
print(f"Vocabulary size (actual): {len(tfidf.vocabulary_):,}")
print(f"X_train_tfidf shape: {X_train_tfidf.shape}")
print(f"X_test_tfidf  shape: {X_test_tfidf.shape}")

# Optional: show top 20 features (alphabetical order by feature name)
feature_names = tfidf.get_feature_names_out()
print("\nSample feature names (first 20):")
print(feature_names[:20])


TF-IDF fitted.
Vocabulary size (actual): 20,000
X_train_tfidf shape: (1600, 20000)
X_test_tfidf  shape: (400, 20000)

Sample feature names (first 20):
['00' '000' '000 years' '007' '10' '10 000' '10 10' '10 minutes'
 '10 scale' '10 things' '10 year' '10 years' '100' '100 000' '100 million'
 '100 minutes' '1000' '100m' '101' '101 dalmatians']


**Observation:** TF‑IDF was fitted with unigrams, English stop words removed, and a hard cap of 20,000 features, producing sparse matrices of shape (1600, 20000) for training and (400, 20000) for testing. The large vocabulary preserves many rare tokens (vocab shows numeric and low‑semantic tokens in the sample), which increases dimensionality and sparsity and can amplify noise or overfitting in downstream classifiers. Considerations and next steps: inspect top features by class, apply frequency thresholds (min_df, max_df) or sublinear_tf, experiment with ngram_range=(1,2) if phrase signals are important, and use regularization or feature selection to control model complexity.

In [45]:
# Define and fit first classifier model (Logistic Regression)

from sklearn.linear_model import LogisticRegression
import time

# Configure model
lr_model = LogisticRegression(
    solver="liblinear",    # good default for small/medium datasets; change to 'saga' or 'lbfgs' if needed
    penalty="l2",
    C=1.0,
    max_iter=1000,
    random_state=42
)

# Fit model and measure time
start_time = time.time()
lr_model.fit(X_train_tfidf, y_train)
fit_time = time.time() - start_time

print(f"Logistic Regression fitted in {fit_time:.2f} seconds.")
# Optionally inspect a few coefficients for top features (requires feature names)
try:
    feature_names = tfidf.get_feature_names_out()
    coefs = lr_model.coef_[0]
    top_pos_idx = coefs.argsort()[-10:][::-1]
    top_neg_idx = coefs.argsort()[:10]
    print("\nTop positive features (indicative of 'pos'):")
    print(", ".join(feature_names[i] for i in top_pos_idx))
    print("\nTop negative features (indicative of 'neg'):")
    print(", ".join(feature_names[i] for i in top_neg_idx))
except Exception:
    # If tfidf or feature names are not available, skip feature inspection
    pass


Logistic Regression fitted in 0.03 seconds.

Top positive features (indicative of 'pos'):
great, perfect, excellent, life, hilarious, world, overall, best, family, perfectly

Top negative features (indicative of 'neg'):
bad, worst, boring, plot, supposed, script, stupid, waste, ridiculous, mess


**Observation**:  
Logistic Regression trained very quickly (~0.2s) on the TF‑IDF features and produced intuitive, high‑weight tokens: positive indicators include words like great, excellent, best, perfect, while negative indicators include bad, worst, boring, stupid. This suggests the TF‑IDF unigram representation captures strong sentiment signals and the classes are reasonably separable. For the notebook: note that while performance is promising, you should still validate with the test set, consider hyperparameter tuning (C, solver), try ngram features or min_df/max_df to reduce noisy tokens, and inspect class‑specific top features to detect dataset artifacts (e.g., proper names or numeric tokens) before finalizing the model.

In [46]:
# Predict test categories with the first classifier (Logistic Regression)
# Compute accuracy, classification report, and show confusion matrix

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np
import pandas as pd

# Predict
y_pred_lr = lr_model.predict(X_test_tfidf)

# Metrics
acc_lr = accuracy_score(y_test, y_pred_lr)
report_lr = classification_report(y_test, y_pred_lr, digits=4)
cm_lr = confusion_matrix(y_test, y_pred_lr, labels=lr_model.classes_)

# Print results
print(f"Logistic Regression Accuracy: {acc_lr:.4f}\n")
print("Classification Report:\n")
print(report_lr)
print("Confusion Matrix (rows=true, cols=predicted) with label order:", list(lr_model.classes_))
print(cm_lr)

# Optional: nicer DataFrame view of confusion matrix
cm_df = pd.DataFrame(cm_lr, index=lr_model.classes_, columns=lr_model.classes_)
print("\nConfusion Matrix (DataFrame):\n")
print(cm_df)


Logistic Regression Accuracy: 0.8425

Classification Report:

              precision    recall  f1-score   support

         neg     0.8586    0.8200    0.8389       200
         pos     0.8278    0.8650    0.8460       200

    accuracy                         0.8425       400
   macro avg     0.8432    0.8425    0.8424       400
weighted avg     0.8432    0.8425    0.8424       400

Confusion Matrix (rows=true, cols=predicted) with label order: ['neg', 'pos']
[[164  36]
 [ 27 173]]

Confusion Matrix (DataFrame):

     neg  pos
neg  164   36
pos   27  173


**Observation**
**Accuracy 0.8225** with balanced class performance: precision/recall/f1 are comparable for both classes (neg: precision 0.8486, **recall 0.7850, f1 0.8156**; **pos: precision 0.8000, recall 0.8600, f1 0.8289)**. The confusion matrix shows **157 true neg / 43 false pos and 172 true pos / 28 false neg**, indicating slightly more false positives than false negatives. Logistic Regression trains quickly and captures strong unigram sentiment signals; next steps in the notebook could include hyperparameter tuning, error analysis on the 71 misclassified examples, trying n‑grams or feature selection, and calibrating probabilities if you need well‑calibrated confidence scores.

In [47]:
# Define and fit second classifier model (Multinomial Naive Bayes)
# Assumes X_train_tfidf, X_test_tfidf, y_train, y_test, and tfidf are defined.

from sklearn.naive_bayes import MultinomialNB
import time
import numpy as np
import pandas as pd

# Configure model
nb_model = MultinomialNB(alpha=1.0)  # Laplace smoothing via alpha

# Fit model and measure time
start_time = time.time()
nb_model.fit(X_train_tfidf, y_train)
fit_time = time.time() - start_time

print(f"MultinomialNB fitted in {fit_time:.2f} seconds.")

# Optional: inspect top features per class (requires feature names)
try:
    feature_names = tfidf.get_feature_names_out()
    class_labels = nb_model.classes_
    # log probability of features per class shape: (n_classes, n_features)
    log_prob = nb_model.feature_log_prob_
    top_k = 10
    for i, cls in enumerate(class_labels):
        top_idx = np.argsort(log_prob[i])[-top_k:][::-1]
        top_feats = [feature_names[j] for j in top_idx]
        print(f"\nTop {top_k} features for class '{cls}':")
        print(", ".join(top_feats))
except Exception:
    pass


MultinomialNB fitted in 0.01 seconds.

Top 10 features for class 'neg':
film, movie, like, just, bad, good, plot, time, story, character

Top 10 features for class 'pos':
film, movie, like, story, good, just, time, life, character, best


In [48]:
# Predict test categories with the second classifier (Multinomial Naive Bayes)

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd

# Predict
y_pred_nb = nb_model.predict(X_test_tfidf)

# Metrics
acc_nb = accuracy_score(y_test, y_pred_nb)
report_nb = classification_report(y_test, y_pred_nb, digits=4)
cm_nb = confusion_matrix(y_test, y_pred_nb, labels=nb_model.classes_)

# Print results
print(f"MultinomialNB Accuracy: {acc_nb:.4f}\n")
print("Classification Report:\n")
print(report_nb)
print("Confusion Matrix (rows=true, cols=predicted) with label order:", list(nb_model.classes_))
print(cm_nb)

# Nicely formatted confusion matrix
cm_df_nb = pd.DataFrame(cm_nb, index=nb_model.classes_, columns=nb_model.classes_)
print("\nConfusion Matrix (DataFrame):\n")
print(cm_df_nb)


MultinomialNB Accuracy: 0.8225

Classification Report:

              precision    recall  f1-score   support

         neg     0.8342    0.8050    0.8193       200
         pos     0.8116    0.8400    0.8256       200

    accuracy                         0.8225       400
   macro avg     0.8229    0.8225    0.8224       400
weighted avg     0.8229    0.8225    0.8224       400

Confusion Matrix (rows=true, cols=predicted) with label order: [np.str_('neg'), np.str_('pos')]
[[161  39]
 [ 32 168]]

Confusion Matrix (DataFrame):

     neg  pos
neg  161   39
pos   32  168


**Observation:** *MultinomialNB* trained extremely quickly (~0.01s) and learned intuitive **unigram signals: both classes share many high‑frequency tokens** (e.g., film, movie, like), while neg is weighted toward words like bad, plot, boring and pos toward good, life, characters. The overlap in top features shows the model mainly exploits subtle weight differences on common tokens rather than distinct rare words. For the notebook: note the speed and interpretability benefits of MultinomialNB, but also consider reducing noisy high‑frequency tokens (use min_df/max_df), adding n‑grams, or applying feature selection to improve discriminative power.

**Limitations and recommendations**
- Lemmatization reduces surface variation but can change meaning (e.g., "what's" → "what be"); mention this in your writeup.
- Add‑one smoothing over a large vocab inflates perplexity for high‑order models; recommend Kneser‑Ney or Katz backoff.
- For generation, prefer sampling from observed continuations or implement backoff to lower orders.
- For TF‑IDF, consider `min_df=2` and `max_df=0.95` or `ngram_range=(1,2)` to reduce noise and capture short phrases.
